# Arabic ABSA Baseline Notebook

This notebook is for **Arabic Aspect-Based Sentiment Analysis (ABSA)**.
A single review can talk about more than one aspect (for example food and service).
Each aspect can have its own sentiment label, so we predict sentiment per aspect, not just one label for the whole review.

This version is made to run locally in Cursor/Jupyter using Excel files in the same folder.

## 1) Import libraries
In this step we import all libraries we need for loading data, cleaning Arabic text, training models, and saving outputs.

In [ ]:
import os
import re
import ast
import json

import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from scipy.sparse import hstack

## 2) Define project constants
Here we set the file names, list of target aspects, and sentiment mapping used in the competition.

In [ ]:
TRAIN_FILE = 'DeepX_train.xlsx'
VAL_FILE = 'DeepX_validation.xlsx'
UNLABELED_FILE = 'DeepX_unlabeled.xlsx'

ASPECTS = [
    'food',
    'service',
    'price',
    'cleanliness',
    'delivery',
    'ambiance',
    'app_experience',
    'general'
]

SENTIMENT_TO_LABEL = {'positive': 1, 'negative': 2, 'neutral': 3}
LABEL_TO_SENTIMENT = {0: 'none', 1: 'positive', 2: 'negative', 3: 'neutral'}

## 3) Load dataset files
Now we load train, validation, and unlabeled files from the current folder, then print basic checks.

In [ ]:
train_df = pd.read_excel(TRAIN_FILE)
val_df = pd.read_excel(VAL_FILE)
unlabeled_df = pd.read_excel(UNLABELED_FILE)

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Unlabeled shape:', unlabeled_df.shape)

print('\nTrain columns:')
print(train_df.columns.tolist())

print('\nTrain sample:')
display(train_df.head(3))

print('\nValidation sample:')
display(val_df.head(3))

## 4) Quick exploratory data analysis (EDA)
We check missing values, and see simple distributions for `platform` and `business_category`.
We also view a few samples from `aspects` and `aspect_sentiments`.

In [ ]:
print('Missing values in train:')
display(train_df.isna().sum())

if 'platform' in train_df.columns:
    print('\nPlatform distribution:')
    display(train_df['platform'].value_counts(dropna=False))

if 'business_category' in train_df.columns:
    print('\nBusiness category distribution:')
    display(train_df['business_category'].value_counts(dropna=False).head(20))

if 'aspects' in train_df.columns:
    print('\nSample aspects values:')
    display(train_df['aspects'].head(5))

if 'aspect_sentiments' in train_df.columns:
    print('\nSample aspect_sentiments values:')
    display(train_df['aspect_sentiments'].head(5))

## 5) Arabic preprocessing function: `remove_tashkeel()`
This function removes Arabic diacritics (tashkeel) so text becomes more consistent.

In [ ]:
def remove_tashkeel(text):
    if not isinstance(text, str):
        text = '' if pd.isna(text) else str(text)
    tashkeel_pattern = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')
    return tashkeel_pattern.sub('', text)

## 6) Arabic preprocessing function: `normalize_arabic()`
This function normalizes common Arabic letter variations (for example: أ/إ/آ to ا).

In [ ]:
def normalize_arabic(text):
    if not isinstance(text, str):
        text = '' if pd.isna(text) else str(text)
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    text = text.replace('ى', 'ي').replace('ة', 'ه')
    return text

## 7) Arabic preprocessing function: `remove_repeated_chars()`
This function shortens repeated characters like `حلوووو` to `حلو`.

In [ ]:
def remove_repeated_chars(text):
    if not isinstance(text, str):
        text = '' if pd.isna(text) else str(text)
    return re.sub(r'(.)\1+', r'\1', text)

## 8) Arabic preprocessing function: `clean_text()`
This function applies all cleaning steps together and removes extra symbols/spaces.

In [ ]:
def clean_text(text):
    text = remove_tashkeel(text)
    text = normalize_arabic(text)
    text = remove_repeated_chars(text)

    # Keep Arabic, English, numbers, and spaces
    text = re.sub(r'[^\u0600-\u06FFa-zA-Z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 9) Apply text cleaning
Now we create a `clean_text` column for train, validation, and unlabeled data.

In [ ]:
for df in [train_df, val_df, unlabeled_df]:
    df['clean_text'] = df['review_text'].apply(clean_text)

display(train_df[['review_text', 'clean_text']].head(3))

## 10) Label parsing function: `safe_parse_list()`
This helper safely parses text into a Python list. If parsing fails, it returns an empty list.

In [ ]:
def safe_parse_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if not isinstance(value, str):
        return []

    text = value.strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

## 11) Label parsing function: `safe_parse_dict()`
This helper safely parses text into a Python dictionary. If parsing fails, it returns an empty dict.

In [ ]:
def safe_parse_dict(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    if not isinstance(value, str):
        return {}

    text = value.strip()
    if not text:
        return {}

    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}

## 12) Label creation function: `create_aspect_labels()`
This function converts raw `aspects` and `aspect_sentiments` into numeric labels for every fixed aspect.

In [ ]:
def create_aspect_labels(aspects_value, aspect_sentiments_value):
    aspects_list = safe_parse_list(aspects_value)
    sentiments_dict = safe_parse_dict(aspect_sentiments_value)

    aspects_set = {str(a).strip().lower() for a in aspects_list}
    sentiments_dict = {str(k).strip().lower(): str(v).strip().lower() for k, v in sentiments_dict.items()}

    labels = {aspect: 0 for aspect in ASPECTS}  # 0 means aspect not present

    for aspect in ASPECTS:
        if aspect in aspects_set:
            sentiment = sentiments_dict.get(aspect, '')
            labels[aspect] = SENTIMENT_TO_LABEL.get(sentiment, 0)

    return labels

## 13) Create training labels for each aspect
We apply `create_aspect_labels()` on train and validation data, then add the 8 aspect columns.

In [ ]:
def add_label_columns(df):
    label_rows = df.apply(lambda row: create_aspect_labels(row['aspects'], row['aspect_sentiments']), axis=1)
    label_df = pd.DataFrame(label_rows.tolist(), index=df.index)
    return pd.concat([df, label_df], axis=1)

train_df = add_label_columns(train_df)
val_df = add_label_columns(val_df)

display(train_df[ASPECTS].head(3))

## 14) Prepare training and validation data
Here we create `X_train`, `X_val`, and the label sets `y_train`, `y_val` for each aspect.

In [ ]:
X_train = train_df['clean_text']
X_val = val_df['clean_text']

y_train = {aspect: train_df[aspect].astype(int) for aspect in ASPECTS}
y_val = {aspect: val_df[aspect].astype(int) for aspect in ASPECTS}

print('X_train size:', X_train.shape)
print('X_val size:', X_val.shape)
print('Aspects prepared:', list(y_train.keys()))

## 15) Word-level TF-IDF
This vectorizer captures normal word n-grams from the cleaned text.

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_val_word = word_vectorizer.transform(X_val)
X_unlabeled_word = word_vectorizer.transform(unlabeled_df['clean_text'])

print('Word TF-IDF train shape:', X_train_word.shape)

## 16) Character-level TF-IDF
This vectorizer captures character n-grams, which helps with spelling variations.

In [ ]:
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    min_df=2
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)
X_unlabeled_char = char_vectorizer.transform(unlabeled_df['clean_text'])

print('Char TF-IDF train shape:', X_train_char.shape)

## 17) Combine word + character features
We combine both feature types using `scipy.sparse.hstack` to build one strong baseline representation.

In [ ]:
X_train_features = hstack([X_train_word, X_train_char])
X_val_features = hstack([X_val_word, X_val_char])
X_unlabeled_features = hstack([X_unlabeled_word, X_unlabeled_char])

print('Combined train feature shape:', X_train_features.shape)

## 18) Train baseline models
We train one Logistic Regression model per aspect and store all models in a dictionary.

In [ ]:
models = {}

for aspect in ASPECTS:
    print(f'Training model for aspect: {aspect}')

    model = LogisticRegression(
        max_iter=500,
        solver='saga',
        multi_class='multinomial',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_features, y_train[aspect])
    models[aspect] = model

print('\nTraining complete. Total models:', len(models))

## 19) Evaluate on validation set
Now we predict the validation set, print a classification report for each aspect, and compute the overall micro F1 score.

In [ ]:
val_predictions = {}

for aspect in ASPECTS:
    y_pred = models[aspect].predict(X_val_features)
    val_predictions[aspect] = y_pred

    print('=' * 80)
    print(f'Aspect: {aspect}')
    print(classification_report(
        y_val[aspect],
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=['none', 'positive', 'negative', 'neutral'],
        zero_division=0
    ))

# Micro F1 across all aspect labels
y_val_all = np.concatenate([y_val[a].values for a in ASPECTS])
y_pred_all = np.concatenate([val_predictions[a] for a in ASPECTS])
micro_f1 = f1_score(y_val_all, y_pred_all, average='micro')
print('Overall micro F1:', round(micro_f1, 4))

### What does F1 mean?
F1 score is a balance between **precision** and **recall**.
In simple words, it checks if the model gives correct labels and also does not miss too many true labels.
A higher F1 score means better overall prediction quality.

## 20) Predict unlabeled data
We predict each aspect for unlabeled reviews, then build `aspects` and `aspect_sentiments`.
If no aspect is found, we use `['none']` and `{'none': 'neutral'}`.

In [ ]:
unlabeled_predictions = {aspect: models[aspect].predict(X_unlabeled_features) for aspect in ASPECTS}

submission_rows = []

for i, row in unlabeled_df.iterrows():
    predicted_aspects = []
    predicted_sentiments = {}

    for aspect in ASPECTS:
        label = int(unlabeled_predictions[aspect][i])
        if label != 0:
            predicted_aspects.append(aspect)
            predicted_sentiments[aspect] = LABEL_TO_SENTIMENT[label]

    if len(predicted_aspects) == 0:
        predicted_aspects = ['none']
        predicted_sentiments = {'none': 'neutral'}

    submission_rows.append({
        'review_id': int(row['review_id']),
        'aspects': predicted_aspects,
        'aspect_sentiments': predicted_sentiments
    })

submission_rows[:2]

## 21) Create `submission.json`
This cell saves predictions in the exact JSON format required by the task.

In [ ]:
with open('submission.json', 'w', encoding='utf-8') as f:
    json.dump(submission_rows, f, ensure_ascii=False, indent=2)

print('submission.json saved successfully.')

## 22) Submission validation function: `check_allowed_aspects()`
This function checks if all predicted aspects are from the allowed list.

In [ ]:
def check_allowed_aspects(aspects_list, allowed_aspects):
    return all(aspect in allowed_aspects for aspect in aspects_list)

## 23) Submission validation function: `check_allowed_sentiments()`
This function checks if all sentiment values are from the allowed sentiment set.

In [ ]:
def check_allowed_sentiments(sentiment_dict, allowed_sentiments):
    return all(sent in allowed_sentiments for sent in sentiment_dict.values())

## 24) Submission validation function: `validate_submission()`
This function runs all submission checks:
- row count
- review_id present
- correct types
- key matching between aspects and aspect_sentiments
- allowed aspect names
- allowed sentiment names

In [ ]:
def validate_submission(submission_data, unlabeled_data):
    allowed_aspects = set(ASPECTS + ['none'])
    allowed_sentiments = {'positive', 'negative', 'neutral'}

    errors = []

    if len(submission_data) != len(unlabeled_data):
        errors.append('Row count does not match unlabeled file.')

    for idx, item in enumerate(submission_data):
        if 'review_id' not in item or pd.isna(item['review_id']):
            errors.append(f'Row {idx}: missing review_id.')

        if not isinstance(item.get('aspects'), list):
            errors.append(f'Row {idx}: aspects is not a list.')
            continue

        if not isinstance(item.get('aspect_sentiments'), dict):
            errors.append(f'Row {idx}: aspect_sentiments is not a dict.')
            continue

        aspects_set = set(item['aspects'])
        sentiment_keys = set(item['aspect_sentiments'].keys())

        if aspects_set != sentiment_keys:
            errors.append(f'Row {idx}: keys in aspect_sentiments do not exactly match aspects.')

        if not check_allowed_aspects(item['aspects'], allowed_aspects):
            errors.append(f'Row {idx}: contains disallowed aspect.')

        if not check_allowed_sentiments(item['aspect_sentiments'], allowed_sentiments):
            errors.append(f'Row {idx}: contains disallowed sentiment.')

    return errors

## 25) Run submission validation
We run the validation function and print whether the file is ready.

In [ ]:
validation_errors = validate_submission(submission_rows, unlabeled_df)

if len(validation_errors) == 0:
    print('Submission validation passed ✅')
else:
    print('Submission validation found issues:')
    for err in validation_errors:
        print('-', err)

## 26) Save vectorizers and models
This step saves trained artifacts in a `models/` folder using joblib so you can reuse them later without retraining.

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(word_vectorizer, 'models/word_tfidf_vectorizer.joblib')
joblib.dump(char_vectorizer, 'models/char_tfidf_vectorizer.joblib')
joblib.dump(models, 'models/aspect_models.joblib')

print('Saved model artifacts in models/')

## 27) Final notes
For submission, keep these files ready:

- `submission.json`
- this notebook file (`.ipynb`)
- `README.md`
- `requirements.txt`
- `models/` folder

This notebook is a simple baseline (TF-IDF + Logistic Regression) and is easy to run from top to bottom in local Jupyter/Cursor.